# Reggie Radiant Reports
Put location of data and access csv files below.

In [1]:
access=r'/kaggle/input/accounts/access.csv' 
data=r'/kaggle/input/accounts/data.csv'

#### Change nothing below:

In [2]:
from datetime import datetime
import time
start=time.time()
date=datetime.today().strftime('%m-%d-%Y')
print("Running Reggie Report on: ", date)

Running Reggie Report on:  08-10-2022


# Access

In [3]:
import pandas as pd

aDF=pd.read_csv(access)
aDF.head()

,VOUCHER#,VOUCHER DATE,PROCESS DATE,INVOICE#,INVOICE DATE,VENDOR,AMOUNT,DESCRIPTION,VOU#,REQ#,ACCOUNT ID,FY
0,V12345,8/8/2022,8/8/2022,JE123456,8/8/2022,WALMART,387.52,Food,PR,Y,20504-05-987000402-123456,FY23
1,V12346,8/9/2022,8/8/2022,JE123457,8/9/2022,TESLA,962.14,Car,PR,Y,20504-05-987000402-987654,FY23
2,V12347,8/10/2022,8/8/2022,JE123458,8/10/2022,AMAZON,17.20,Stuff,PR,Y,20504-05-987000402-123456,FY23
3,V12348,8/11/2022,8/8/2022,JE123459,8/11/2022,AMAZON,26.38,Computer,PR,Y,20504-05-987000402-888999,FY23
4,V12349,8/12/2022,8/8/2022,JE123460,8/12/2022,TACO,45.00,NaN,PR,Y,98204-05-987000402-777666,FY23


#### Scan for Duplicate Entries

In [4]:
duplicateRows = aDF[aDF.duplicated(['VOUCHER#', 'AMOUNT','ACCOUNT ID', 'INVOICE#'])]
print(duplicateRows['VOUCHER#'])

8    V12352
Name: VOUCHER#, dtype: object


#### Filter Important Columns

In [5]:
aDF=aDF[['AMOUNT', 'ACCOUNT ID', 'REQ#']]
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,20504-05-987000402-123456,Y
1,962.14,20504-05-987000402-987654,Y
2,17.20,20504-05-987000402-123456,Y
3,26.38,20504-05-987000402-888999,Y
4,45.00,98204-05-987000402-777666,Y


#### Remove Hypens

In [6]:
aDF['ACCOUNT ID'] = aDF['ACCOUNT ID'].astype(str).apply(lambda x: x.replace('-', ''))
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,2050405987000402123456,Y
1,962.14,2050405987000402987654,Y
2,17.20,2050405987000402123456,Y
3,26.38,2050405987000402888999,Y
4,45.00,9820405987000402777666,Y


#### Find only POSTED access records

In [7]:
aDF=aDF[aDF['REQ#'].str.contains('Y')]
aDF.head()

,AMOUNT,ACCOUNT ID,REQ#
0,387.52,2050405987000402123456,Y
1,962.14,2050405987000402987654,Y
2,17.20,2050405987000402123456,Y
3,26.38,2050405987000402888999,Y
4,45.00,9820405987000402777666,Y


#### Add each amount by each accounting total

In [8]:
aDF=aDF.groupby('ACCOUNT ID')['AMOUNT'].sum().reset_index().round(2)
aDF.head()

,ACCOUNT ID,AMOUNT
0,2050405987000402123456,2293.95
1,2050405987000402888999,26.38
2,2050405987000402987654,962.14
3,9820405987000402444333,265.50
4,9820405987000402777666,45.00


# Data Transactions

In [9]:
df=pd.read_csv(data)
df.head()

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,NaN
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,NaN
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,NaN
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,NaN
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,NaN


#### Fill empty YTD Credit with 0

In [10]:
df[' YTD Credits']=df[' YTD Credits'].fillna(0)
df.head()

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,0.0
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,0.0
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,0.0
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,0.0
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,0.0


#### Debits minus credits

In [11]:
df = df.assign(DataTotal = df[' YTD Debits'] - df[' YTD Credits'])
df.head()                                                   

,GL Account,GL Description,Date,Source,Reference Number,Description,Opening Balance,YTD Debits,YTD Credits,DataTotal
0,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6100,PR,NaN,387.52,0.0,387.52
1,20504-05-987000402-987654,ReggieCR,8/8/2022,JE,AAVCDF6101,PR,NaN,962.14,0.0,962.14
2,20504-05-987000402-123456,ReggieCR,8/8/2022,JE,AAVCDF6102,PR,NaN,17.20,0.0,17.20
3,20504-05-987000402-888999,ReggieCR,8/8/2022,JE,AAVCDF6103,PR,NaN,26.38,0.0,26.38
4,98204-05-987000402-777666,ReggieCR,8/8/2022,JE,AAVCDF6104,PR,NaN,45.00,0.0,45.00


#### Filter by Account Numbers and Amounts

In [12]:
df=df[['GL Account', 'DataTotal']]
df.head()

,GL Account,DataTotal
0,20504-05-987000402-123456,387.52
1,20504-05-987000402-987654,962.14
2,20504-05-987000402-123456,17.20
3,20504-05-987000402-888999,26.38
4,98204-05-987000402-777666,45.00


#### Remove Hypens (temporarily)

In [13]:
df['GL Account'] = df['GL Account'].astype(str).apply(lambda x: x.replace('-', ''))
df.head()

,GL Account,DataTotal
0,2050405987000402123456,387.52
1,2050405987000402987654,962.14
2,2050405987000402123456,17.20
3,2050405987000402888999,26.38
4,9820405987000402777666,45.00


In [14]:
df = df.groupby('GL Account')['DataTotal'].sum().reset_index().round(2)
df.head()

,GL Account,DataTotal
0,1234567987654000111333,52.35
1,1234567987654000555555,56.42
2,2050405987000402123456,2946.18
3,2050405987000402888999,26.38
4,2050405987000402987654,962.14


#### Check for new Data account number

In [15]:
dataAccounts=list(df['GL Account'])
accessAccounts=list(aDF['ACCOUNT ID'])
for d in dataAccounts:
    if d not in accessAccounts:
        print(d, " account not in access")

1234567987654000111333  account not in access
1234567987654000555555  account not in access


# Join Data and Access

#### Rename Access Account ID to GL Account to do a join

In [16]:
aDF=aDF.rename(columns={"ACCOUNT ID": "GL Account"})
df.head()

,GL Account,DataTotal
0,1234567987654000111333,52.35
1,1234567987654000555555,56.42
2,2050405987000402123456,2946.18
3,2050405987000402888999,26.38
4,2050405987000402987654,962.14


#### Join Data by Account Number

In [17]:
finalDF=pd.merge(df, aDF, on='GL Account')
finalDF=finalDF.rename(columns={"AMOUNT":"AccessAmount"})
finalDF.head()

,GL Account,DataTotal,AccessAmount
0,2050405987000402123456,2946.18,2293.95
1,2050405987000402888999,26.38,26.38
2,2050405987000402987654,962.14,962.14
3,9820405987000402444333,265.50,265.50
4,9820405987000402777666,45.00,45.00


#### Add Hypens again
1. Easier to copy paste account numbers later 
2. New account numbers will not turn into scientific notation if turned back into an excel sheet

In [18]:
accHyphens=[]
for a in finalDF['GL Account']:
    accHyphens.append(a[0:5]+'-'+a[6:15]+'-'+a[16:22])
    
name = df.columns[0] # Find the name of the column by index
finalDF.drop(name, axis = 1, inplace = True) #drop column
finalDF.insert(0, name, accHyphens ) #readd column that has hyphens
finalDF.head()

,GL Account,DataTotal,AccessAmount
0,20504-598700040-123456,2946.18,2293.95
1,20504-598700040-888999,26.38,26.38
2,20504-598700040-987654,962.14,962.14
3,98204-598700040-444333,265.50,265.50
4,98204-598700040-777666,45.00,45.00


#### Calculate Difference and then sort by priority 

In [19]:
finalDF = finalDF.assign(Difference = finalDF['DataTotal'] - finalDF['AccessAmount']).sort_values(by=['Difference'], ascending=False).reset_index(drop=True)
finalDF.head()

,GL Account,DataTotal,AccessAmount,Difference
0,20504-598700040-123456,2946.18,2293.95,652.23
1,98204-598700040-990099,2215.04,2210.64,4.40
2,20504-598700040-888999,26.38,26.38,0.00
3,20504-598700040-987654,962.14,962.14,0.00
4,98204-598700040-444333,265.50,265.50,0.00


#### Create Excel Sheet

In [20]:
finalDF.to_csv("ReggieReport.csv", index=False)

# Create Excel sheet with Header

In [21]:
'''
import xlsxwriter

text1 = "Campus Department"
text2 = "Reggie Report ran on: " + date

writer = pd.ExcelWriter('ReggieReport.xlsx', engine='xlsxwriter')

# Write each dataframe to a different worksheet.
finalDF.to_excel(writer, sheet_name='name', startrow=3, startcol=2, header=True, index=False)

worksheet1 = writer.sheets['name']

#Write to each worksheet:
worksheet1.write(0, 0, text1)
worksheet1.write(1, 0, text2)

# Close the Pandas Excel writer and output the Excel file.
writer.save()
print("Done")
'''

'\nimport xlsxwriter\n\ntext1 = "Campus Department"\ntext2 = "Reggie Report ran on: " + date\n\nwriter = pd.ExcelWriter(\'ReggieReport.xlsx\', engine=\'xlsxwriter\')\n\n# Write each dataframe to a different worksheet.\nfinalDF.to_excel(writer, sheet_name=\'name\', startrow=3, startcol=2, header=True, index=False)\n\nworksheet1 = writer.sheets[\'name\']\n\n#Write to each worksheet:\nworksheet1.write(0, 0, text1)\nworksheet1.write(1, 0, text2)\n\n# Close the Pandas Excel writer and output the Excel file.\nwriter.save()\nprint("Done")\n'

In [22]:
print("Done. Finished in ", round((time.time()-start), 4), " seconds")

Done. Finished in  1.2088  seconds
